# HoHoNet 质量分析交互式 Notebook

本 Notebook 用于交互式分析 HoHoNet 标注质量报告，支持：
- 单个 `quality_report_*.csv` 文件加载
- 与 `aggregate_analysis.py` 输出对接（多数据集比较）
- 可视化探索与异常检测
- 批量导出图表（可选）

**使用前准备**：
1. 确保已安装依赖：`pandas`, `numpy`, `matplotlib`, `seaborn`, `ipywidgets`
2. 运行 `analyze_quality.py` 生成 quality_report CSV
3. （可选）运行 `aggregate_analysis.py` 合并多数据集

**快速开始**：逐个运行下面的代码单元格即可。

## 1. 导入库和配置环境

In [ ]:
# 导入必要的库
import sys
import os
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from IPython.display import display, Markdown

# 添加 tools 目录到 sys.path (如果在其他目录运行)
tools_dir = Path(__file__).parent if '__file__' in globals() else Path.cwd()
if str(tools_dir) not in sys.path:
    sys.path.insert(0, str(tools_dir))

# 导入自定义工具模块
from viz_quality_utils import (
    load_quality_report,
    compute_task_summary,
    compute_disagreement_table,
    filter_in_scope,
    filter_layout_used,
    build_summary_stats,
    plot_scope_distribution,
    plot_layout_gate_reasons,
    plot_metric_distribution,
    plot_metric_vs_time,
    plot_metric_by_annotator,
    plot_time_by_annotator,
    plot_task_annotator_heatmap,
    plot_mixed_scope_tasks,
)

# 设置 seaborn 样式和颜色
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']  # 中文字体支持
plt.rcParams['axes.unicode_minus'] = False  # 负号显示

# 全局配置
DEFAULT_METRIC = "iou"  # 默认分析指标
COLOR_PALETTE = sns.color_palette("Set2")

print("✅ 环境配置完成！")

## 2. 数据加载与预处理

支持两种加载方式：
- **方式 A**：加载单个 `quality_report_*.csv`
- **方式 B**：加载 `aggregate_analysis.py` 输出的合并 CSV（自动识别 `dataset`/`condition`/`subset` 列）

In [ ]:
# === 配置：选择你的 CSV 文件路径 ===
CSV_PATH = "../analysis_results/quality_report_20260116.csv"  # 修改为你的路径

# 加载数据
print(f"加载数据: {CSV_PATH}")
df = load_quality_report(CSV_PATH)

print(f"\n✅ 数据加载完成！")
print(f"  - 总行数: {len(df)}")
print(f"  - 任务数: {df['task_id'].nunique()}")
print(f"  - 标注者: {sorted(df['annotator_id'].unique())}")
print(f"  - 列数: {len(df.columns)}")

# 显示前几行
display(df.head(3))

## 3. 与 aggregate_analysis 对接（可选）

如果你使用了 `aggregate_analysis.py` 合并多数据集，可以加载它生成的 JSON 摘要：

In [ ]:
# === 可选：加载 aggregate_analysis 的 summary JSON ===
AGG_JSON_PATH = "../analysis_results/aggregate_20260116_summary.json"  # 修改为你的路径

if os.path.exists(AGG_JSON_PATH):
    with open(AGG_JSON_PATH, "r", encoding="utf-8") as f:
        agg_summary = json.load(f)
    
    print("✅ 已加载 aggregate 摘要：")
    print(f"  - 数据集数量: {len(agg_summary.get('metadata', []))}")
    print(f"  - 分组数: {len(agg_summary.get('summary_by_group', {}))}")
    
    # 提取 cross-dataset 比较表
    comparisons = agg_summary.get("cross_dataset_comparisons", {})
    if comparisons:
        print("\n跨数据集对比:")
        for comp_name, metrics in comparisons.items():
            print(f"  {comp_name}:")
            for metric, vals in metrics.items():
                print(f"    {metric}: Manual={vals['manual']:.4f}, Semi={vals['semi']:.4f}, Diff={vals['diff']:.4f}")
else:
    print(f"⚠️  未找到 aggregate summary JSON: {AGG_JSON_PATH}")
    agg_summary = None

## 4. 数据质量概览

In [ ]:
# 生成数据质量摘要
stats = build_summary_stats(df)

# 格式化为 Markdown
md_lines = ["### 数据质量摘要\n"]
md_lines.append(f"- **总行数**: {stats['n_rows']}")
md_lines.append(f"- **任务数**: {stats['n_tasks']}")
md_lines.append(f"- **标注者**: {', '.join(stats['n_annotators'])}\n")

md_lines.append("#### 每任务标注覆盖率")
md_lines.append("| 标注者数 | 任务数 |")
md_lines.append("|---------|--------|")
for n, count in stats["annotators_per_task"].items():
    md_lines.append(f"| {n} | {count} |")

if "layout_used_rate" in stats:
    md_lines.append(f"\n#### Layout Gating")
    md_lines.append(f"- **layout_used 比例**: {stats['layout_used_rate']:.2%}")
    if stats.get("layout_gate_reasons"):
        md_lines.append("\n**拒绝原因分布**:")
        for reason, cnt in stats["layout_gate_reasons"].items():
            md_lines.append(f"- {reason}: {cnt}")

if "scope_empty_count" in stats:
    md_lines.append(f"\n#### Scope 卫生检查")
    md_lines.append(f"- **scope 缺失行数**: {stats['scope_empty_count']}/{stats['n_rows']}")

if stats.get("mixed_scope_tasks"):
    md_lines.append(f"\n⚠️ **Mixed-scope 任务**: {', '.join(map(str, stats['mixed_scope_tasks']))}")

display(Markdown("\n".join(md_lines)))

## 5. Scope 分布分析

In [ ]:
# 绘制 scope 分布
fig = plot_scope_distribution(df, figsize=(10, 4))
plt.show()

# 统计 OOS 比例
oos_count = df["scope"].str.lower().str.startswith("oos").sum()
print(f"\n📊 Out-of-Scope 比例: {oos_count}/{len(df)} ({oos_count/len(df):.1%})")

## 6. Layout Gating 分析

In [ ]:
# 绘制 layout_gate_reason 分布
if "layout_used" in df.columns:
    fig = plot_layout_gate_reasons(df, figsize=(10, 4))
    if fig:
        plt.show()
    else:
        print("ℹ️  所有样本的 layout_used=True，无门控拒绝")

## 7. 指标分布可视化

In [ ]:
# === 配置：选择要分析的指标 ===
METRIC = "iou"  # 可选: iou, layout_2d_iou, layout_3d_iou, boundary_rmse_px 等

# 绘制指标分布（按 scope 分层）
fig = plot_metric_distribution(df, metric=METRIC, hue="scope", bins=25)
plt.show()

# 基本统计
print(f"\n📊 {METRIC} 统计:")
print(df[METRIC].describe())

## 8. 时间效率分析

In [ ]:
# 绘制 metric vs active_time 散点图
if "active_time" in df.columns:
    fig = plot_metric_vs_time(df, metric=METRIC, hue="scope", style="annotator_id")
    plt.show()
    
    # 识别异常耗时任务
    q99 = df["active_time"].quantile(0.99)
    outliers = df[df["active_time"] > q99].sort_values("active_time", ascending=False)
    print(f"\n⚠️  异常耗时任务 (> P99 = {q99:.1f}s):")
    display(outliers[["task_id", "annotator_id", "active_time", METRIC, "scope"]].head(10))

## 9. 标注者间比较

In [ ]:
# 绘制标注者的指标箱线图
fig = plot_metric_by_annotator(df, metric=METRIC)
plt.show()

# 绘制标注者的时间箱线图
if "active_time" in df.columns:
    fig = plot_time_by_annotator(df)
    plt.show()

# 按标注者分组统计
annotator_stats = df.groupby("annotator_id")[METRIC].agg(["mean", "median", "std", "count"])
print(f"\n📊 标注者 {METRIC} 统计:")
display(annotator_stats)

## 10. 任务级热力图（发现分歧）

In [ ]:
# 绘制 task × annotator 热力图
fig = plot_task_annotator_heatmap(df, metric=METRIC, max_tasks=60)
plt.show()

# 计算任务级分歧表
disagreement = compute_disagreement_table(df, metric=METRIC)
print(f"\n⚠️  Top 10 分歧最大的任务 ({METRIC} variance 最高):")
display(disagreement.head(10))

## 11. 任务级聚合（一行一个任务）

In [ ]:
# 生成任务级汇总 DataFrame
task_summary = compute_task_summary(df, metric_cols=[METRIC, "boundary_rmse_px", "active_time"])

print(f"✅ 任务级汇总生成完成！({len(task_summary)} 个任务)")
display(task_summary.head(10))

# 导出为 CSV（可选）
# task_summary.to_csv("../analysis_results/task_summary.csv", index=False)
# print("📁 已保存到 task_summary.csv")

## 12. 异常样本识别

In [ ]:
# 识别低 IoU 样本
low_metric = df.sort_values(METRIC).head(20)
print(f"⚠️  Top 20 最低 {METRIC} 样本:")
display(low_metric[["task_id", "annotator_id", METRIC, "boundary_rmse_px", "active_time", "scope", "difficulty"]])

# 识别高边界误差样本
if "boundary_rmse_px" in df.columns:
    high_boundary = df.sort_values("boundary_rmse_px", ascending=False).head(20)
    print(f"\n⚠️  Top 20 最高 boundary_rmse_px 样本:")
    display(high_boundary[["task_id", "annotator_id", METRIC, "boundary_rmse_px", "scope"]])

## 13. 批量导出图表（可选）

如果需要批量保存图表，使用 `save_quality_figures.py` 脚本：

```bash
python tools/save_quality_figures.py analysis_results/quality_report_20260116.csv \
    --out-dir analysis_results/figures \
    --tag pilot_20260116 \
    --metric iou
```

或者在 notebook 里手动保存当前图表：

In [ ]:
# 示例：保存当前工作区的所有图表
from datetime import datetime

EXPORT_DIR = Path("../analysis_results/figures") / f"notebook_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# 你可以在每个绘图 cell 后添加：
# plt.savefig(EXPORT_DIR / "plot_name.png", dpi=200, bbox_inches="tight")

print(f"✅ 图表导出目录已创建: {EXPORT_DIR}")

## 总结与下一步

本 Notebook 提供了 HoHoNet 质量分析的基础工具集，你可以根据需求自定义：

- **筛选数据**：使用 `filter_in_scope(df)` 或 `filter_layout_used(df)` 过滤子集
- **更换指标**：修改 `METRIC` 变量分析其他列（如 `layout_2d_iou`, `layout_depth_rmse`）
- **交互探索**：结合 `ipywidgets` 添加下拉菜单/滑块
- **对接 aggregate**：加载合并 CSV 后按 `dataset`/`condition` 分组对比

**相关脚本**：
- `viz_quality_utils.py`：可复用的函数库
- `save_quality_figures.py`：批量生成图表
- `aggregate_analysis.py`：多数据集合并与对比

**文档**：
- [tools/README.md](tools/README.md)：工具链详细说明
- [docs/HANDOVER_2026-01-16.md](docs/HANDOVER_2026-01-16.md)：项目交接文档